<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/Updates/Second_Test_WA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **STEP 1: CLEAN DATA**

# Import Extensions And Open File

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import joblib
from collections import defaultdict
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')

# Remove Unnecessary Columns

In [ ]:
# List of columns to remove
columns_to_remove = [

    # Identifier columns
    'Customer Id',
    'Category Id',
    'Department Id',

    # Personal/customer detail columns
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',
    'Customer Zipcode',
    'Customer Full Name',

    # High-cardinality text columns
    'Product Description',
    'Product Image',
    'Product Name',


    # Duplicate or Redundant Geographic Columns

'Customer Segment',
'Order City',
'Order State',]

# Remove only columns that actually exist in dataframe
existing_columns_to_remove = [
    col for col in columns_to_remove if col in df.columns]

# Drop columns
df = df.drop(columns=existing_columns_to_remove)

# Display remaining columns
print("Remaining Columns:")
print(df.columns.tolist())

# Display shape after removal
print("\nNew Dataset Shape:")
print(df.shape)

Remaining Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer State', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'Shipping Mode']

New Dataset Shape:
(180519, 38)


# Identify missing or erroneous data


In [ ]:

print(df.shape)
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

(180519, 38)
Order Zipcode    155679
dtype: int64


In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate Rows: {duplicate_count}")

Duplicate Rows: 0


In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0])

Order Zipcode    155679
dtype: int64


In [ ]:
num_cols = df.select_dtypes(include=['int64','float64']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
    cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
non_negative_columns = [
    'Sales',
    'Order Item Quantity',
    'Order Item Total',
    'Product Price',
    'Days for shipping (scheduled)']

for col in non_negative_columns:

    if col in df.columns:

        # Count invalid negatives
        invalid_count = (df[col] < 0).sum()

        print(f"\n{col} Negative Values: {invalid_count}")

        # Replace negatives with median
        median_value = df[df[col] >= 0][col].median()

        df.loc[df[col] < 0, col] = median_value


Sales Negative Values: 0

Order Item Quantity Negative Values: 0

Order Item Total Negative Values: 0

Product Price Negative Values: 0


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

outlier_summary = {}

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    outlier_summary[col] = outliers

outlier_df = pd.DataFrame.from_dict(
    outlier_summary,
    orient='index',
    columns=['Outlier Count']
)

print(outlier_df.sort_values('Outlier Count', ascending=False))

                               Outlier Count
Order Zipcode                          24818
Benefit per order                      18942
Order Profit Per Order                 18942
Order Item Profit Ratio                17300
Order Item Discount                     7537
Order Item Product Price                2048
Product Price                           2048
Sales per customer                      1943
Order Item Total                        1943
Longitude                               1414
Order Customer Id                       1198
Sales                                    488
Latitude                                   9
Late_delivery_risk                         0
Days for shipment (scheduled)              0
Days for shipping (real)                   0
Order Item Quantity                        0
Order Item Id                              0
Order Item Cardprod Id                     0
Order Item Discount Rate                   0
Order Id                                   0
Product Ca

# Winsorize Outliers

In [ ]:
for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])

# Log Transform Highly Skewed Variables

In [ ]:
skew_cols = [
    'Order Item Total',
    'Order Profit Per Order',
    'Shipping Cost',
    'Profit Per Unit'
]

for col in skew_cols:
    if col in df.columns:
        df[col] = np.log1p(df[col] - df[col].min())

In [ ]:
print("\nRemaining Missing Values:")
print(df.isnull().sum().sum())

print("\nFinal Dataset Shape:")
print(df.shape)

print("\nDataset Preview:")
print(df.head())


Remaining Missing Values:
0

Final Dataset Shape:
(180519, 38)

Dataset Preview:
       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                       3.0                            4.0   
1  TRANSFER                       5.0                            4.0   
2      CASH                       4.0                            4.0   
3     DEBIT                       3.0                            4.0   
4   PAYMENT                       2.0                            4.0   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1         -79.700005          311.359985     Late delivery   
2         -79.700005          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk   Category Name Customer City Customer Country  ...  \
0                 0.0  Sporting Goo

# Remove Constant and Near-Constant Features

In [ ]:
constant_cols = [
    col for col in df.columns
    if df[col].nunique() == 1
]

print("Constant Features:")
print(constant_cols)

df.drop(columns=constant_cols, inplace=True)
near_constant_cols = []

for col in df.columns:

    freq = df[col].value_counts(normalize=True)

    # Check if freq is empty (e.g., column was entirely NaN) or if a single value dominates
    if freq.empty or freq.iloc[0] > 0.99:
        near_constant_cols.append(col)

print("Near Constant Features:")
print(near_constant_cols)

df.drop(columns=near_constant_cols, inplace=True)

Constant Features:
['Order Zipcode', 'Product Status']
Near Constant Features:
[]


# Remove Duplicate Rows

In [ ]:
print("Rows before:", len(df))

df.drop_duplicates(inplace=True)

print("Rows after:", len(df))

Rows before: 180519
Rows after: 180519


# Verify Dataset Quality

In [ ]:
print("Remaining Missing Values:")
print(df.isnull().sum().sum())

print("Dataset Shape:")
print(df.shape)

Remaining Missing Values:
0
Dataset Shape:
(180519, 36)


# Recompute Numeric Columns

In [ ]:
numeric_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()

print(f"Number of numeric features: {len(numeric_cols)}")
print(numeric_cols)

Number of numeric features: 22
['Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Late_delivery_risk', 'Latitude', 'Longitude', 'Order Customer Id', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Product Card Id', 'Product Category Id', 'Product Price']


#Save Data

In [ ]:
# Save cleaned RL dataset to CSV in Google Colab

output_file = 'dataco_rl_cleaned.csv'

# Save dataframe
df.to_csv(output_file, index=False)

print(f"Dataset saved as: {output_file}")

Dataset saved as: dataco_rl_cleaned.csv


# **STEP 2: FEATURE ENGINEERING**

In [ ]:
df = pd.read_csv('dataco_rl_cleaned.csv', encoding='latin1')

print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(180519, 36)


# Date and Time Features

In [ ]:
# Convert order date to datetime
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce')

# Extract temporal features
df['order_year'] = df['order date (DateOrders)'].dt.year
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_day'] = df['order date (DateOrders)'].dt.day
df['order_dayofweek'] = df['order date (DateOrders)'].dt.dayofweek
df['is_weekend'] = df['order_dayofweek'].isin([5,6]).astype(int)


# Create a Product-Region Matrix and Network

In [ ]:
# Create a Product-Region Matrix
product_region = (
    df.groupby("Product Category Id")["Order Region"]
      .nunique()
      .reset_index(name="num_regions")
      .sort_values("num_regions", ascending=False))

In [ ]:
order_features = (
    df.groupby("Order Id")
      .agg(
          unique_products=("Product Category Id", "nunique"),
          total_quantity=("Order Item Quantity", "sum"),
          total_sales=("Sales", "sum"))
      .reset_index())

order_features["complex_order"] = (
    order_features["unique_products"] >= 3
).astype(int)

In [ ]:
network = (
    df.groupby(["Product Category Id", "Order Region"])
      .agg(
          orders=("Order Id", "nunique"),
          quantity=("Order Item Quantity", "sum"),
          revenue=("Sales", "sum"))
      .reset_index())

# Explore Product Statistics

In [ ]:
product_stats = (
    network.groupby("Product Category Id")
           .agg(
               regions_served=("Order Region", "nunique"),
               total_orders=("orders", "sum"),
               total_quantity=("quantity", "sum"),
               total_revenue=("revenue", "sum"))
           .reset_index())


# Create Shipping Mode Statistics

In [ ]:
import pandas as pd
import numpy as np

# Ensure 'Shipping Mode_str' and 'Order Region_str' are available
df['Shipping Mode_str'] = df['Shipping Mode']
df['Order Region_str'] = df['Order Region']

df = df[df['Order Region_str'] == 'West Africa'].copy()

# Historical shipping mode performance
mode_stats = (
    df.groupby('Shipping Mode_str')
      .agg({
          'Order Profit Per Order':'mean',
          'Late_delivery_risk':'mean',
          'Days for shipping (real)':'mean'
      })
      .reset_index()
)

print(mode_stats)

  Shipping Mode_str  Order Profit Per Order  Late_delivery_risk  \
0       First Class                4.368462            0.978402   
1          Same Day                4.236680            0.354260   
2      Second Class                4.310360            0.772358   
3    Standard Class                4.396597            0.374560   

   Days for shipping (real)  
0                  2.000000  
1                  0.390135  
2                  3.979675  
3                  4.010563  


# Define Multi-Region Stock

In [ ]:
region_threshold = product_stats["regions_served"].median()
order_threshold = product_stats["total_orders"].median()

product_stats["likely_multi_region_stocked"] = (
    (product_stats["regions_served"] >= region_threshold)
    &
    (product_stats["total_orders"] >= order_threshold)
).astype(int)

#Create a stock score

In [ ]:
product_stats["stocking_score"] = (
    0.5 *
    (
        product_stats["regions_served"]
        / product_stats["regions_served"].max())
    +
    0.5 *
    (
        product_stats["total_orders"]
        / product_stats["total_orders"].max()))


# Merge the network

In [ ]:
network = network.merge(
    product_stats[
        [
            "Product Category Id",
            "regions_served",
            "stocking_score",
            "likely_multi_region_stocked"
        ]
    ],
    on="Product Category Id",
    how="left")

# Find Edge Weight

In [ ]:
network["edge_weight"] = (
    0.4 *
    (network["orders"] / network["orders"].max())
    +
    0.3 *
    (network["quantity"] / network["quantity"].max())
    +
    0.3 *
    (network["revenue"] / network["revenue"].max()))


 # Find candidate shipping regions

In [ ]:
candidate_regions = (
    network.sort_values(
        ["Product Category Id", "edge_weight"],
        ascending=[True, False])
    .groupby("Product Category Id")
    .head(5))

# Save Outputs

In [ ]:
network.to_csv(
    "product_region_network.csv",
    index=False
)

candidate_regions.to_csv(
    "candidate_fulfillment_regions.csv",
    index=False
)

product_stats.to_csv(
    "product_stocking_scores.csv",
    index=False
)

print("Files created successfully")

Files created successfully


# Shipping Engineering

In [ ]:
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce'
)
df['shipping date (DateOrders)'] = pd.to_datetime(
    df['shipping date (DateOrders)'],
    errors='coerce'
)

# Actual shipping time

df['actual_ship_days'] = (
    df['shipping date (DateOrders)']
    - df['order date (DateOrders)']
).dt.days

# Difference from scheduled

df['shipping_delay'] = (
    df['Days for shipping (real)']
    - df['Days for shipment (scheduled)']
)

# Binary on-time flag

df['on_time_delivery'] = (
    df['shipping_delay'] <= 0
).astype(int)

# Late shipment flag

df['late_delivery'] = (
    df['shipping_delay'] > 0
).astype(int)

# Profit Engineering

In [ ]:
# Margin percentage

df['margin_pct'] = (
    df['Order Profit Per Order']
    / df['Sales']
)

# Profit per item

df['profit_per_item'] = (
    df['Order Profit Per Order']
    / df['Order Item Quantity']
)

# Discount percentage

df['discount_pct'] = (
    df['Order Item Discount']
    / df['Order Item Product Price']
)

# Product Demand Features

In [ ]:
product_stats = (
    df.groupby('Product Category Id')
      .agg(
          product_orders=('Order Id','nunique'),
          product_quantity=('Order Item Quantity','sum'),
          product_revenue=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    product_stats,
    on='Product Category Id',
    how='left'
)

# Regional Features

In [ ]:
region_stats = (
    df.groupby('Order Region')
      .agg(
          region_orders=('Order Id','nunique'),
          region_sales=('Sales','sum'),
          region_profit=('Order Profit Per Order','sum')
      )
      .reset_index()
)

df = df.merge(
    region_stats,
    on='Order Region',
    how='left'
)

# Fulfillment Network Features

In [ ]:
stocking = pd.read_csv(
    "product_stocking_scores.csv", encoding='latin1'
)

df = df.merge(
    stocking[
        [
            'Product Category Id',
            'stocking_score',
            'regions_served',
            'likely_multi_region_stocked'
        ]
    ],
    on='Product Category Id',
    how='left'
)

# Shipping Mode Features

In [ ]:
speed_mapping = {
    'Same Day':4,
    'First Class':3,
    'Second Class':2,
    'Standard Class':1
}
df['shipping_mode_speed'] = df['Shipping Mode'].map(speed_mapping)

# Create string versions of Shipping Mode and Order Region for later use
df['Shipping Mode_str'] = df['Shipping Mode']
df['Order Region_str'] = df['Order Region']

action_id_mapping = {
    'Standard Class': 0,    # Maps to key 0 in shipping_cost_map
    'Second Class': 1,     # Maps to key 1 in shipping_cost_map
    'First Class': 2,      # Maps to key 2 in shipping_cost_map
    'Same Day': 3          # Maps to key 3 in shipping_cost_map
}
df['action_id'] = df['Shipping Mode'].map(action_id_mapping)

shipping_cost_map = {
    0: 1.0,    # Standard
    1: 1.5,    # Second
    2: 2.0,    # First
    3: 3.0     # Same Day
}

BASE_COST = 10

df['Mode_Cost'] = (
    BASE_COST *
    df['action_id'].map(shipping_cost_map)
)

# Order Complexity Features

In [ ]:
order_complexity = (
    df.groupby('Order Id')
      .agg(
          unique_products=('Product Category Id','nunique'),
          total_quantity=('Order Item Quantity','sum'),
          total_sales=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    order_complexity,
    on='Order Id',
    how='left'
)

df['complex_order'] = (
    df['unique_products'] >= 3
).astype(int)

# Route-Based Shipping Costs

In [ ]:
df = df.copy()

# Ensure 'Shipping Cost' is present, assuming it should align with 'Mode_Cost'
df['Shipping Cost'] = df['Mode_Cost']

df["Route"] = (
    df["Order Region"].astype(str)
    + " -> "
    + df["Order Country"].astype(str)
)

df["Shipping Cost"] = df["Mode_Cost"]

route_mode_cost = (
    df.groupby(["Route", "Shipping Mode"])["Shipping Cost"]
    .mean()
    .reset_index()
    .rename(columns={"Shipping Cost": "Route_Mode_Cost"})
)

df = df.merge(
    route_mode_cost,
    on=["Route", "Shipping Mode"],
    how="left"
)

route_mode_cost = (
    df.groupby(
        ['Route','Shipping Mode']
    )['Shipping Cost']
    .mean()
    .reset_index()
)

route_mode_cost.rename(
    columns={'Shipping Cost':'Route_Mode_Cost'},
    inplace=True
)

df = df.merge(
    route_mode_cost,
    on=['Route','Shipping Mode'],
    how='left'
)

# RL State Features

In [ ]:
state_features = [
    'Product Category Id',
    'Order Region',
    'Market',
    'Shipping Mode',
    'stocking_score',
    'regions_served',
    'likely_multi_region_stocked',
    'unique_products',
    'total_quantity',
    'shipping_delay',
    'margin_pct',
    'profit_per_item',
    'order_month',
    'order_dayofweek'
]

# Reward Features

In [ ]:
df['reward'] = (
    df['Order Profit Per Order']
    - df['Route_Mode_Cost_y']
    - 10 * df['Late_delivery_risk']
)

# Save Engineered Dataset

In [ ]:
df.to_csv(
    "dataco_rl_feature_engineered.csv",
    index=False
)

print(df.shape)

(3696, 71)


# **STEP 3: ENCODING CATEGORICAL VARIABLES**

In [ ]:

df = pd.read_csv('dataco_rl_feature_engineered.csv', encoding='latin1')


print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(3696, 71)


In [ ]:
one_hot_cols = [
    'Shipping Mode',
    'Market',
    'Order Region'
]

df = pd.get_dummies(
    df,
    columns=one_hot_cols,
    drop_first=False
)

df.to_csv(
    "dataco_rl_onehot.csv",
    index=False
)

print(df.shape)

(3696, 74)


# Filter on West Africa

In [ ]:
df = pd.read_csv("dataco_rl_onehot.csv", encoding = 'latin1')

# Keep only West Africa orders
west_africa_df = df[df["Order Region_West Africa"] == 1].copy()

print("Number of West Africa orders:", len(west_africa_df))

Number of West Africa orders: 3696


# **STEP 4: TRAIN/TEST TEMPORAL SPLIT**

In [ ]:
import pandas as pd

df = pd.read_csv("dataco_rl_onehot.csv", encoding = 'latin1')

df = df[df["Order Region_West Africa"] == 1].copy()

df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)']
)

df = df.sort_values(
    'order date (DateOrders)'
)

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(2587, 74)
(554, 74)
(555, 74)


In [ ]:
# Save datasets

train_df.to_csv(
    "dataco_rl_train.csv",
    index=False
)

val_df.to_csv(
    "dataco_rl_validation.csv",
    index=False
)

test_df.to_csv(
    "dataco_rl_test.csv",
    index=False
)

print("Files saved successfully")

Files saved successfully


# STEP 5: STATE-SPACE DISCRETIZATION

# Feature Selection

In [ ]:
state_features = [
    'Profit Margin',
    'Shipping Cost Ratio',
    'Historical On-Time Rate',
    'Regional Delay Score',
    'Product Category Id'
]

# Fit Bins Using Training Data Only

In [ ]:
train_df = pd.read_csv("dataco_rl_train.csv", encoding, 'latin1')

# Use train_df instead of df for binning operations
train_df['shipping_cost_bin'] = pd.qcut(
    train_df['Shipping Cost'],
    q=5,
    labels=False,
    duplicates='drop'
)

train_df['order_value_bin'] = pd.qcut(
    train_df['Order Item Total'],
    q=5,
    labels=False,
    duplicates='drop'
)

# The 'Order Region' column does not exist in train_df.
# Since train_df is already filtered to 'West Africa' and 'Order Region'
# was one-hot encoded, 'Order Region_bin' will be a constant.
# Assigning a constant value to create the expected column.
train_df['Order Region_bin'] = 0

exclude_cols = [
    'Reward',
    'Late_delivery_risk',
    'Action_ID'
]

continuous_features = [
    col for col in train_df.select_dtypes(include='number').columns
    if col not in exclude_cols
    and not col.endswith('_bin') # Exclude the newly created bin columns from being binned again
]

bin_edges = {}

# Create 10 bins for each feature
for col in continuous_features:
    # Handle cases where all values are the same to prevent qcut errors
    if train_df[col].nunique() == 1:
        # If all values are the same, create a single bin from min to min (or min to min + epsilon)
        min_val = train_df[col].min()
        bins = np.array([min_val, min_val + 1e-9]) # Add a small epsilon to ensure two distinct points
    else:
        _, bins = pd.qcut(
            train_df[col],
            q=10,
            labels=False,
            retbins=True,
            duplicates='drop'
        )
    bin_edges[col] = bins

# Save bin definitions
joblib.dump(
    bin_edges,
    "state_bins.pkl"
)

print("Bins created successfully")

Bins created successfully


In [ ]:
bin_counts = {}

for col in continuous_features:
    _, bins = pd.qcut(
        train_df[col],
        q=10,
        retbins=True,
        duplicates='drop'
    )

    bin_counts[col] = len(bins) - 1

print(bin_counts)

{'Days for shipping (real)': 5, 'Days for shipment (scheduled)': 3, 'Benefit per order': 10, 'Sales per customer': 10, 'Latitude': 10, 'Longitude': 10, 'Order Customer Id': 10, 'Order Id': 10, 'Order Item Cardprod Id': 9, 'Order Item Discount': 10, 'Order Item Discount Rate': 10, 'Order Item Id': 10, 'Order Item Product Price': 8, 'Order Item Profit Ratio': 10, 'Order Item Quantity': 4, 'Sales': 9, 'Order Item Total': 10, 'Order Profit Per Order': 10, 'Product Card Id': 9, 'Product Category Id': 9, 'Product Price': 8, 'order_year': 0, 'order_month': 4, 'order_day': 10, 'order_dayofweek': 6, 'is_weekend': 1, 'actual_ship_days': 5, 'shipping_delay': 5, 'on_time_delivery': 1, 'late_delivery': 1, 'margin_pct': 10, 'profit_per_item': 10, 'discount_pct': 10, 'product_orders': 8, 'product_quantity': 9, 'product_revenue': 8, 'region_orders': 0, 'region_sales': 0, 'region_profit': 0, 'stocking_score': 8, 'regions_served': 1, 'likely_multi_region_stocked': 1, 'shipping_mode_speed': 3, 'action_id

# Apply Bins to Train/Test Data

In [ ]:
train_df = pd.read_csv("dataco_rl_train.csv", encoding = 'latin1')
test_df = pd.read_csv("dataco_rl_test.csv", encoding = 'latin1')

bin_edges = joblib.load("state_bins.pkl")

continuous_features = list(bin_edges.keys())

for col in continuous_features:

    train_df[col + "_bin"] = np.digitize(
        train_df[col],
        bin_edges[col][1:-1]
    )

    test_df[col + "_bin"] = np.digitize(
        test_df[col],
        bin_edges[col][1:-1]
    )

train_df.to_csv(
    "dataco_rl_train_discrete.csv",
    index=False
)

test_df.to_csv(
    "dataco_rl_test_discrete.csv",
    index=False
)

print("Discretization complete")

Discretization complete


# Build Action Lookup Table

In [ ]:
from sklearn.preprocessing import LabelEncoder
import joblib
import pandas as pd
import numpy as np


# Load split files

train_df = pd.read_csv("dataco_rl_train_discrete.csv", encoding = 'latin1')

val_df = pd.read_csv("dataco_rl_validation.csv", encoding = 'latin1')
test_df = pd.read_csv("dataco_rl_test.csv", encoding = 'latin1')


# Load candidate fulfillment regions

candidate_regions = pd.read_csv("candidate_fulfillment_regions.csv", encoding = 'latin1')

# Rename fulfillment region column if needed
if "Order Region" in candidate_regions.columns:
    candidate_regions = candidate_regions.rename(
        columns={"Order Region": "Fulfillment_Region"}
    )

# Keep only needed columns
candidate_regions = candidate_regions[
    ["Product Category Id", "Fulfillment_Region"]
].drop_duplicates()


# Merge fulfillment regions into train/val/test

train_df = train_df.merge(
    candidate_regions,
    on="Product Category Id",
    how="left"
)

val_df = val_df.merge(
    candidate_regions,
    on="Product Category Id",
    how="left"
)

test_df = test_df.merge(
    candidate_regions,
    on="Product Category Id",
    how="left"
)


# Restore Shipping Mode Text
shipping_mode_map_reverse = {
    1: "Standard Class",
    2: "Second Class",
    3: "First Class",
    4: "Same Day"
}

for df_iter in [train_df, val_df, test_df]:
    if "Shipping Mode_str" not in df_iter.columns:
        df_iter["Shipping Mode_str"] = df_iter["shipping_mode_speed"].map(
            shipping_mode_map_reverse
        )


# Restore order region text

for df_iter in [train_df, val_df, test_df]:
    if "Order Region_str" not in df_iter.columns:
        df_iter["Order Region_str"] = "West Africa"
    # Add Order Region_bin manually, as it's a constant for West Africa
    df_iter['Order Region_bin'] = 0

# Re-apply binning to val_df and test_df after loading non-discrete versions

bin_edges = joblib.load("state_bins.pkl")
continuous_features = list(bin_edges.keys())

for df_to_bin in [val_df, test_df]:
    for col in continuous_features:
        if col in df_to_bin.columns:
            if len(bin_edges[col]) > 2:
                df_to_bin[col + "_bin"] = np.digitize(
                    df_to_bin[col],
                    bin_edges[col][1:-1]
                )
            else:
                df_to_bin[col + "_bin"] = 0



# Create route

for df_iter in [train_df, val_df, test_df]:
    df_iter["Route"] = (
        df_iter["Order Region_str"].astype(str)
        + " -> "
        + df_iter["Order Country"].astype(str)
    )


# Create combined action
# Fulfillment Region + Route + Shipping Mode

for df_iter in [train_df, val_df, test_df]:
    df_iter["action"] = (
        df_iter["Fulfillment_Region"].astype(str)
        + " | "
        + df_iter["Route"].astype(str)
        + " | "
        + df_iter["Shipping Mode_str"].astype(str)
    )


# Create state IDs

state_cols = [
    "Product Category Id_bin",
    "stocking_score_bin",
    "margin_pct_bin",
    "shipping_delay_bin",
    "Order Region_bin",
    "Shipping Cost_bin",
    "Order Item Total_bin"
]

# Check for missing state columns in train_df first
missing_train_state_cols = [col for col in state_cols if col not in train_df.columns]
if missing_train_state_cols:
    raise ValueError(f"Missing state columns in train_df: {missing_train_state_cols}")

train_df["state"] = train_df[state_cols].astype(str).agg("|".join, axis=1)

state_encoder = LabelEncoder()
train_df["state_id"] = state_encoder.fit_transform(train_df["state"])


# Create action IDs

action_encoder = LabelEncoder()
train_df["action_id"] = action_encoder.fit_transform(train_df["action"])

# Handle unseen actions/states in validation/test
valid_actions = set(action_encoder.classes_)
valid_states = set(state_encoder.classes_)

# Process val_df and test_df for state and action IDs
for df_proc in [val_df, test_df]:

    missing_proc_state_cols = [col for col in state_cols if col not in df_proc.columns]
    if missing_proc_state_cols:

        raise ValueError(f"Missing state columns in processed dataframe: {missing_proc_state_cols}")

    df_proc["state"] = df_proc[state_cols].astype(str).agg("|".join, axis=1)


    df_proc = df_proc[df_proc["action"].isin(valid_actions)].copy()
    df_proc = df_proc[df_proc["state"].isin(valid_states)].copy()

    df_proc["action_id"] = action_encoder.transform(df_proc["action"])
    df_proc["state_id"] = state_encoder.transform(df_proc["state"])


# Collect processed dataframes
processed_val_df = None
processed_test_df = None

# Re-process val_df and test_df correctly to update external variables
val_df["state"] = val_df[state_cols].astype(str).agg("|".join, axis=1)
val_df = val_df[val_df["action"].isin(valid_actions)].copy()
val_df = val_df[val_df["state"].isin(valid_states)].copy()
val_df["action_id"] = action_encoder.transform(val_df["action"])
val_df["state_id"] = state_encoder.transform(val_df["state"])

test_df["state"] = test_df[state_cols].astype(str).agg("|".join, axis=1)
test_df = test_df[test_df["action"].isin(valid_actions)].copy()
test_df = test_df[test_df["state"].isin(valid_states)].copy()
test_df["action_id"] = action_encoder.transform(test_df["action"])
test_df["state_id"] = state_encoder.transform(test_df["state"])


# Save encoders

joblib.dump(state_encoder, "state_encoder.pkl")
joblib.dump(action_encoder, "action_encoder.pkl")


# Final checks

n_states = train_df["state_id"].nunique()
n_actions = train_df["action_id"].nunique()

print("Rows:", len(train_df))
print("States:", n_states)
print("Actions:", n_actions)

print("\nSample actions:")
print(train_df["action"].head(10))

print("\nAction encoder classes:")
print(action_encoder.classes_[:10])

Rows: 12935
States: 1011
Actions: 335

Sample actions:
0    Central America | West Africa -> Ghana | Secon...
1    Western Europe | West Africa -> Ghana | Second...
2    South America | West Africa -> Ghana | Second ...
3    Southern Europe | West Africa -> Ghana | Secon...
4    Northern Europe | West Africa -> Ghana | Secon...
5    Central America | West Africa -> Ghana | Secon...
6    Western Europe | West Africa -> Ghana | Second...
7    South America | West Africa -> Ghana | Second ...
8    Northern Europe | West Africa -> Ghana | Secon...
9    Southern Europe | West Africa -> Ghana | Secon...
Name: action, dtype: object

Action encoder classes:
['Caribbean | West Africa -> BenÃ\x83Â\xadn | First Class'
 'Caribbean | West Africa -> BenÃ\x83Â\xadn | Second Class'
 'Caribbean | West Africa -> BenÃ\x83Â\xadn | Standard Class'
 'Caribbean | West Africa -> Burkina Faso | Standard Class'
 'Caribbean | West Africa -> Costa de Marfil | First Class'
 'Caribbean | West Africa -> Costa de Mar

In [ ]:
print("Rows:", len(train_df))
print("States:", train_df["state_id"].nunique())
print("Actions:", train_df["action_id"].nunique())
print(train_df["action"].head())

Rows: 12935
States: 1011
Actions: 335
0    Central America | West Africa -> Ghana | Secon...
1    Western Europe | West Africa -> Ghana | Second...
2    South America | West Africa -> Ghana | Second ...
3    Southern Europe | West Africa -> Ghana | Secon...
4    Northern Europe | West Africa -> Ghana | Secon...
Name: action, dtype: object


In [ ]:
print("Q shape should be:")
print("States:", train_df["state_id"].nunique())
print("Actions:", train_df["action_id"].nunique())

n_states = train_df["state_id"].nunique()
n_actions = train_df["action_id"].nunique()

Q = np.zeros((n_states, n_actions))

print(Q.shape)

Q shape should be:
States: 1011
Actions: 335
(1011, 335)


# Check State Space Size

In [ ]:
print("States:", train_df["state_id"].nunique())
print("Actions:", train_df["action_id"].nunique())

States: 1011
Actions: 335


In [ ]:
print(
    "Unique states:",
    train_df['state'].nunique()
)

print(
    "Unique actions:",
    train_df['action'].nunique()
)

print(
    "Q-table size:",
    train_df['state'].nunique()
    *
    train_df['action'].nunique()
)

Unique states: 1011
Unique actions: 335
Q-table size: 338685


# **STEP 6: Q-LEARNING IMPLEMENTATION**


# Dual-Goal Reward Fuction

In [ ]:
PROFIT_WEIGHT = 5.0
LATE_PENALTY = 8.0
TIME_PENALTY = 0.10
COST_PENALTY = 1.0

train_df["reward"] = (
    PROFIT_WEIGHT * train_df["Order Profit Per Order"]
    - COST_PENALTY * train_df["Route_Mode_Cost_y"]
    - LATE_PENALTY * train_df["Late_delivery_risk"]
    - TIME_PENALTY * train_df["Days for shipping (real)"]
)
n_states = train_df["state_id"].nunique()
n_actions = train_df["action_id"].nunique()

# Initialize Q-Table and Train

In [ ]:
Q = np.zeros((n_states, n_actions))

alpha = 0.1
gamma = 0.5
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
episodes = 100

reward_table = (
    train_df.groupby(["state_id", "action_id"])["reward"]
    .mean()
    .to_dict()
)

state_ids = train_df["state_id"].to_numpy()
mean_reward = train_df["reward"].mean()

for episode in range(episodes):
    total_reward = 0

    for i in range(len(state_ids) - 1):
        state = int(state_ids[i])
        next_state = int(state_ids[i + 1])

        if np.random.rand() < epsilon:
            action = np.random.randint(n_actions)
        else:
            action = np.argmax(Q[state])

        reward = reward_table.get((state, action), mean_reward)

        Q[state, action] += alpha * (
            reward + gamma * np.max(Q[next_state]) - Q[state, action]
        )

        total_reward += reward

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if episode % 10 == 0:
        print(f"Episode {episode}: Reward = {total_reward:.2f}")

print("Training complete")

Episode 0: Reward = 49403.84
Episode 10: Reward = 51248.82
Episode 20: Reward = 53174.24
Episode 30: Reward = 56097.81
Episode 40: Reward = 57821.85
Episode 50: Reward = 59598.86
Episode 60: Reward = 61235.06
Episode 70: Reward = 63014.51
Episode 80: Reward = 64016.26
Episode 90: Reward = 66640.00
Training complete


In [ ]:
policy = np.argmax(Q, axis=1)

policy_df = pd.DataFrame({
    "state_id": np.arange(n_states),
    "recommended_action": action_encoder.inverse_transform(policy)
})

policy_df.head(10)

,state_id,recommended_action
0,0,Southeast Asia | West Africa -> Nigeria | Same...
1,1,Northern Europe | West Africa -> Liberia | Sta...
2,2,Southern Europe | West Africa -> NÃÂ­ger | St...
3,3,Caribbean | West Africa -> Costa de Marfil | S...
4,4,Oceania | West Africa -> Nigeria | First Class
5,5,Western Europe | West Africa -> Costa de Marfi...
6,6,Southeast Asia | West Africa -> Guinea | Stand...
7,7,South America | West Africa -> Guinea | Standa...
8,8,Central America | West Africa -> BenÃÂ­n | St...
9,9,Southern Europe | West Africa -> Costa de Marf...


# Extract the Learned Policy

In [ ]:
policy_action_ids = np.argmax(Q, axis=1)

policy_df = pd.DataFrame({
    "state_id": np.arange(Q.shape[0]),
    "action_id": policy_action_ids,
    "recommended_action": action_encoder.inverse_transform(policy_action_ids)
})

# Split combined action into readable columns
policy_df[
    [
        "Recommended_Fulfillment_Region",
        "Recommended_Route",
        "Recommended_Shipping_Mode"
    ]
] = policy_df["recommended_action"].str.split(" \\| ", expand=True)

policy_df.head(10)

,state_id,action_id,recommended_action,Recommended_Fulfillment_Region,Recommended_Route,Recommended_Shipping_Mode
0,0,225,Southeast Asia | West Africa -> Nigeria | Same...,Southeast Asia,West Africa -> Nigeria,Same Day
1,1,96,Northern Europe | West Africa -> Liberia | Sta...,Northern Europe,West Africa -> Liberia,Standard Class
2,2,263,Southern Europe | West Africa -> NÃÂ­ger | St...,Southern Europe,West Africa -> NÃÂ­ger,Standard Class
3,3,7,Caribbean | West Africa -> Costa de Marfil | S...,Caribbean,West Africa -> Costa de Marfil,Standard Class
4,4,143,Oceania | West Africa -> Nigeria | First Class,Oceania,West Africa -> Nigeria,First Class
5,5,297,Western Europe | West Africa -> Costa de Marfi...,Western Europe,West Africa -> Costa de Marfil,Second Class
6,6,218,Southeast Asia | West Africa -> Guinea | Stand...,Southeast Asia,West Africa -> Guinea,Standard Class
7,7,173,South America | West Africa -> Guinea | Standa...,South America,West Africa -> Guinea,Standard Class
8,8,35,Central America | West Africa -> BenÃÂ­n | St...,Central America,West Africa -> BenÃÂ­n,Standard Class
9,9,236,Southern Europe | West Africa -> Costa de Marf...,Southern Europe,West Africa -> Costa de Marfil,Same Day


In [ ]:
# Extract best action for each state
policy = np.argmax(Q, axis=1)

# Convert encoded actions back to route/shipping-mode labels
policy_actions = action_encoder.inverse_transform(policy)

# View first few recommendations
print(policy_actions[:10])

['Southeast Asia | West Africa -> Nigeria | Same Day'
 'Northern Europe | West Africa -> Liberia | Standard Class'
 'Southern Europe | West Africa -> NÃ\x83Â\xadger | Standard Class'
 'Caribbean | West Africa -> Costa de Marfil | Standard Class'
 'Oceania | West Africa -> Nigeria | First Class'
 'Western Europe | West Africa -> Costa de Marfil | Second Class'
 'Southeast Asia | West Africa -> Guinea | Standard Class'
 'South America | West Africa -> Guinea | Standard Class'
 'Central America | West Africa -> BenÃ\x83Â\xadn | Standard Class'
 'Southern Europe | West Africa -> Costa de Marfil | Same Day']


In [ ]:
policy_df.to_csv("learned_policy.csv", index=False)
print("Saved learned_policy.csv")

Saved learned_policy.csv


# Evaluate Recommended Shipping Modes

In [ ]:
# Extract best action for each state
policy = np.argmax(Q, axis=1)

# Convert encoded actions back to route/shipping-mode labels
policy_actions = action_encoder.inverse_transform(policy)

# Add recommended action to each state
policy_df = pd.DataFrame({
    "state_id": np.arange(len(policy_actions)),
    "recommended_action": policy_actions
})

# Merge recommendations back to training data
eval_df = train_df.merge(policy_df, on="state_id", how="left")

# If action contains route/mode like:
# Fulfillment_Region | Route | Shipping_Mode
eval_df["Recommended_Shipping_Mode"] = (
    eval_df["recommended_action"]
    .astype(str)
    .str.split("|")
    .str[-1]
    .str.strip()
)

# Compare historical vs recommended shipping mode
mode_comparison = pd.crosstab(
    eval_df["Shipping Mode_str"],
    eval_df["Recommended_Shipping_Mode"],
    normalize="index"
) * 100

print("Historical vs Recommended Shipping Mode (%):")
display(mode_comparison.round(2))

# Recommended mode distribution
recommended_distribution = (
    eval_df["Recommended_Shipping_Mode"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Recommended Shipping Mode Distribution (%):")
display(recommended_distribution)

# Historical mode distribution
historical_distribution = (
    eval_df["Shipping Mode_str"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Historical Shipping Mode Distribution (%):")
display(historical_distribution)

Historical vs Recommended Shipping Mode (%):


Recommended_Shipping_Mode,First Class,Same Day,Second Class,Standard Class
Shipping Mode_str,,,,
First Class,18.72,10.34,27.37,43.58
Same Day,25.00,14.53,22.09,38.37
Second Class,16.36,13.04,36.85,33.75
Standard Class,7.81,3.68,11.25,77.26


Recommended Shipping Mode Distribution (%):


,proportion
Recommended_Shipping_Mode,
Standard Class,61.89
Second Class,18.98
First Class,12.06
Same Day,7.07


Historical Shipping Mode Distribution (%):


,proportion
Shipping Mode_str,
Standard Class,60.84
Second Class,18.67
First Class,13.84
Same Day,6.65


In [ ]:
recommended_mode_summary = (
    eval_df.groupby("Recommended_Shipping_Mode")
    .agg(
        orders=("Order Id", "count"),
        avg_profit=("Order Profit Per Order", "mean"),
        avg_late_risk=("Late_delivery_risk", "mean"),
        avg_shipping_days=("Days for shipping (real)", "mean"),
        avg_route_mode_cost=("Route_Mode_Cost_y", "mean"),
        avg_reward=("reward", "mean")
    )
    .sort_values("avg_reward", ascending=False)
)

display(recommended_mode_summary.round(4))

,orders,avg_profit,avg_late_risk,avg_shipping_days,avg_route_mode_cost,avg_reward
Recommended_Shipping_Mode,,,,,,
Standard Class,8005,4.5004,0.4047,3.4728,12.3079,6.6090
Second Class,2455,4.1858,0.6843,3.4501,15.3564,-0.2469
First Class,1560,4.2903,0.7500,3.3974,16.1699,-1.0581
Same Day,915,4.0801,0.6721,3.0601,16.4754,-1.7578


# Measure Improvement

In [ ]:
metrics = {
    "Avg Profit Per Order": "Order Profit Per Order",
    "Late Delivery Risk": "Late_delivery_risk",
    "Avg Shipping Days": "Days for shipping (real)",
    "Avg Route/Mode Cost": "Route_Mode_Cost_y",
    "Avg Reward": "reward"
}

results = []

for metric_name, col in metrics.items():
    historical = eval_df[col].mean()

    policy = (
        eval_df
        .groupby("Recommended_Shipping_Mode")[col]
        .transform("mean")
        .mean()
    )

    change = policy - historical

    pct_change = (change / historical) * 100 if historical != 0 else np.nan

    results.append({
        "Metric": metric_name,
        "Historical": historical,
        "Recommended Policy": policy,
        "Change": change,
        "Percent Change": pct_change
    })

improvement_df = pd.DataFrame(results)

display(improvement_df.round(4))

,Metric,Historical,Recommended Policy,Change,Percent Change
0,Avg Profit Per Order,4.3856,4.3856,0.0,0.0
1,Late Delivery Risk,0.5184,0.5184,0.0,0.0
2,Avg Shipping Days,3.4302,3.4302,0.0,0.0
3,Avg Route/Mode Cost,13.6471,13.6471,0.0,0.0
4,Avg Reward,3.7912,3.7912,0.0,0.0


In [ ]:
state_best_action = {
    s: policy_actions[s]
    for s in range(len(policy_actions))
}

eval_df["recommended_action"] = (
    eval_df["state_id"]
    .map(state_best_action)
)

In [ ]:
different_pct = (
    eval_df["action"]
    != eval_df["recommended_action"]
).mean() * 100

print(
    f"Orders changed by policy: "
    f"{different_pct:.2f}%"
)

Orders changed by policy: 96.40%


In [ ]:
eval_df["recommended_action"] = (
    eval_df["state_id"]
    .map(dict(enumerate(policy_actions)))
)

changed_pct = (
    eval_df["action"]
    != eval_df["recommended_action"]
).mean() * 100

print(f"Orders changed by policy: {changed_pct:.2f}%")

Orders changed by policy: 96.40%


In [ ]:
# Historical action value from Q-table
eval_df["historical_q"] = [
    Q[s, a] for s, a in zip(eval_df["state_id"], eval_df["action_id"])
]

# Recommended policy value
eval_df["policy_q"] = [
    Q[s].max() for s in eval_df["state_id"]
]

q_comparison = pd.DataFrame({
    "Metric": ["Avg Q-Value"],
    "Historical": [eval_df["historical_q"].mean()],
    "Recommended Policy": [eval_df["policy_q"].mean()]
})

q_comparison["Change"] = (
    q_comparison["Recommended Policy"] - q_comparison["Historical"]
)

q_comparison["Percent Change"] = (
    q_comparison["Change"] / q_comparison["Historical"].replace(0, np.nan)
) * 100

display(q_comparison.round(4))

,Metric,Historical,Recommended Policy,Change,Percent Change
0,Avg Q-Value,5.3073,14.872,9.5647,180.219


In [ ]:
comparison = pd.crosstab(
    eval_df["Shipping Mode_str"],
    eval_df["recommended_action"].str.split("|").str[-1].str.strip(),
    normalize="index"
) * 100

display(comparison.round(2))

recommended_action,First Class,Same Day,Second Class,Standard Class
Shipping Mode_str,,,,
First Class,18.72,10.34,27.37,43.58
Same Day,25.00,14.53,22.09,38.37
Second Class,16.36,13.04,36.85,33.75
Standard Class,7.81,3.68,11.25,77.26


In [ ]:
pd.Series(policy_actions).value_counts(normalize=True).head(20) * 100

,proportion
South America | West Africa -> Nigeria | Standard Class,2.868447
Northern Europe | West Africa -> Nigeria | Standard Class,2.472799
Central America | West Africa -> Nigeria | Standard Class,2.472799
Western Europe | West Africa -> Nigeria | Standard Class,1.879327
Northern Europe | West Africa -> Senegal | Standard Class,1.681503
Southern Europe | West Africa -> Nigeria | Standard Class,1.088032
Central America | West Africa -> Senegal | Standard Class,1.088032
South America | West Africa -> Ghana | Standard Class,0.989120
Western Europe | West Africa -> Senegal | Standard Class,0.890208
Western Europe | West Africa -> Ghana | Standard Class,0.890208


# Save Results

In [ ]:
policy_df.to_csv(
    "q_learning_policy.csv",
    index=False
)

np.save(
    "q_table.npy",
    Q
)

print("Policy and Q-table saved")

Policy and Q-table saved


# **STEP 7: SARSA IMPLEMENTATION**


# **STEP 8: COMPARISON OF PROFIT, REWARD, AND ON-TIME DELIVERY PERFORMANCE**


* State = information known BEFORE shipment decision
* Action = shipping method / route / priority decision
* Reward = profit − lateness penalty
* Next state = next order environment

Reward Function

Rt=α(Profit)−β(Late Shipment Penalty)

Where:

α controls profit importance

β controls delivery performance importance

A more detailed version:

Rt=αPt−βLt−γCt

Where:
Pt = profit

Lt = lateness indicator or delay days

Ct = shipping cost

# STEP 8: Q-learning or SARSA environment setup